# Prediksi Arah Pergerakan Harian USD/IDR dengan Machine Learning

**Jenis masalah:** klasifikasi biner. Target = 1 jika kurs USD/IDR besok **lebih tinggi** dari hari ini, 0 jika turun atau tetap.

**Alur notebook (sesuai Ketentuan Pengerjaan):**

| No | Tahap | Isi |
|---|---|---|
| 1 | Persiapan | Import library, baca data USD/IDR, JCI, dan DXY |
| 2 | EDA | Ringkasan data (`info`, `describe`) |
| 3 | Data cleaning | Missing value, duplikat, outlier, encoding. Setiap keputusan disertai alasan |
| 4 | EDA lanjutan | Distribusi target dan 4 visualisasi, masing-masing dengan temuan |
| 5 | Feature engineering | Fitur yang hanya memakai informasi sampai hari ini |
| 6 | Split data | `train_test_split(test_size=0.2, random_state=42, stratify=y)` |
| 7 | Model standar | `KNeighborsClassifier` dengan scaling (pembanding) |
| 8 | Model lain | Logistic Regression, Random Forest, XGBoost, LightGBM, dengan aturan yang sama |
| 9 | Pemilihan model | Cross-validation 5 lipatan dan tuning pada data latih saja |
| 10 | Evaluasi akhir | Data uji dipakai **satu kali** untuk model terpilih. Accuracy, precision, recall, confusion matrix |
| 11 | Interpretasi | Arti metrik, alasan memilih model, keterbatasan |

## 1. Persiapan

In [ ]:
# Jalankan hanya jika library belum terpasang (misalnya di Google Colab)
!pip -q install xgboost lightgbm

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
def cari_file(nama):
    """Cari file di /content (Colab) lalu di folder kerja saat ini."""
    for base in ["/content", "."]:
        path = os.path.join(base, nama)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"File tidak ditemukan: {nama}")

usd_raw = pd.read_csv(cari_file("USD_IDR Historical Data (5).csv"))
jci_raw = pd.read_csv(cari_file("Jakarta Stock Exchange Composite Index Historical Data (2).csv"))
dxy_raw = pd.read_csv(cari_file("US Dollar Index Historical Data (3).csv"))

print("USD/IDR:", usd_raw.shape)
print("JCI    :", jci_raw.shape)
print("DXY    :", dxy_raw.shape)
display(usd_raw.head())

## 2. EDA: Ringkasan Data

Ringkasan dibuat pada data **mentah** (sebelum cleaning) supaya masalah tipe data dan missing value terlihat.

In [ ]:
for nama, df in [("USD/IDR", usd_raw), ("JCI", jci_raw), ("DXY", dxy_raw)]:
    print("=" * 70)
    print(nama)
    df.info()

print("\nStatistik deskriptif USD/IDR (data mentah):")
display(usd_raw.describe(include="all").T)

**Temuan ringkasan data**

- Semua kolom harga (`Price`, `Open`, `High`, `Low`) bertipe `object` karena memakai pemisah ribuan koma (contoh `17,892.8`). Kolom `Change %` juga `object` karena ada simbol `%`, dan `Date` masih berupa teks. Semuanya harus dikonversi sebelum dianalisis.
- Kolom `Vol.` pada USD/IDR dan DXY sebagian besar kosong. Pada JCI, `Vol.` berupa teks dengan akhiran `B` (miliar).
- Data USD/IDR tercatat dari Oktober 2015 sampai Oktober 2026 (sekitar 2.800 baris harian).

## 3. Data Cleaning

Setiap keputusan ditulis beserta alasannya.

### 3.1 Konversi tipe data

**Keputusan:** `Date` diubah ke `datetime`, kolom harga dan `Change %` diubah ke angka, data diurutkan dari tanggal terlama.
**Alasan:** model dan perhitungan return/rolling hanya bisa memakai angka, dan fitur berbasis waktu (rolling, shift) mensyaratkan urutan kronologis.

In [ ]:
def konversi_tipe(df):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
    for col in ["Price", "Open", "High", "Low"]:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False), errors="coerce")
    df["Change %"] = pd.to_numeric(df["Change %"].astype(str).str.replace("%", "", regex=False), errors="coerce")
    return df.sort_values("Date").reset_index(drop=True)

usd = konversi_tipe(usd_raw)
jci = konversi_tipe(jci_raw)
dxy = konversi_tipe(dxy_raw)

print("Tanggal gagal di-parse (NaT):", usd["Date"].isna().sum(), jci["Date"].isna().sum(), dxy["Date"].isna().sum())
print(usd.dtypes)

### 3.2 Missing value

In [ ]:
for nama, df in [("USD/IDR", usd), ("JCI", jci), ("DXY", dxy)]:
    miss = df.isna().sum().to_frame("Jumlah Missing")
    miss["Persen"] = (miss["Jumlah Missing"] / len(df) * 100).round(2)
    print(nama)
    display(miss)

**Keputusan 1:** kolom `Vol.` **dibuang** dari ketiga dataset.
**Alasan:** pada USD/IDR dan DXY sebagian besar nilainya kosong (kurs dan indeks valuta tidak punya volume terpusat), sehingga mengisinya berarti mengarang data. Pada JCI hanya 3 baris kosong, tetapi volume tidak tersedia untuk dua variabel lain, jadi dibuang agar konsisten.

**Keputusan 2:** kolom `Change %` **tidak dipakai sebagai fitur**.
**Alasan:** nilainya hanya pembulatan dari return harian yang akan dihitung sendiri dari `Price`, sehingga redundan.

**Keputusan 3:** harga JCI dan DXY yang kosong pada tanggal tertentu (hari libur bursa di salah satu pasar) **diisi dengan nilai terakhir yang tersedia (forward fill)** setelah digabung dengan USD/IDR.
**Alasan:** saat pasar tutup, harga terakhir memang masih menjadi harga yang berlaku. Forward fill hanya memakai informasi masa lalu, jadi tidak membocorkan data masa depan. Baris awal yang masih kosong setelah ffill dibuang.

In [ ]:
usd = usd.drop(columns=["Vol.", "Change %"])
jci = jci.drop(columns=["Vol.", "Change %"])
dxy = dxy.drop(columns=["Vol.", "Change %"])
print("Sisa missing USD/IDR:", int(usd.isna().sum().sum()))

### 3.3 Duplikat

In [ ]:
for nama, df in [("USD/IDR", usd), ("JCI", jci), ("DXY", dxy)]:
    print(f"{nama}: baris duplikat penuh = {df.duplicated().sum()}, tanggal duplikat = {df['Date'].duplicated().sum()}")

usd = usd.drop_duplicates(subset="Date", keep="last").reset_index(drop=True)
jci = jci.drop_duplicates(subset="Date", keep="last").reset_index(drop=True)
dxy = dxy.drop_duplicates(subset="Date", keep="last").reset_index(drop=True)
print("Jumlah baris USD/IDR setelah pengecekan duplikat:", len(usd))

**Keputusan:** duplikat diperiksa berdasarkan baris penuh dan berdasarkan `Date`; bila ada, hanya satu baris per tanggal yang disimpan (`keep="last"`).
**Alasan:** satu tanggal harus mewakili satu observasi. Duplikat membuat satu hari terhitung dua kali, dan setelah split acak salinannya bisa jatuh ke data latih dan data uji sekaligus sehingga skor uji menjadi terlalu optimistis.

### 3.4 Outlier

Outlier diperiksa pada **return harian**, bukan pada harga, karena harga naik turun mengikuti tren jangka panjang dan tidak cocok diukur dengan IQR.

In [ ]:
usd["Return_%"] = usd["Price"].pct_change() * 100
r = usd["Return_%"].dropna()

q1, q3 = r.quantile([0.25, 0.75])
iqr = q3 - q1
batas_bawah, batas_atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
mask_outlier = (usd["Return_%"] < batas_bawah) | (usd["Return_%"] > batas_atas)

print(f"Batas IQR: {batas_bawah:.3f}% sampai {batas_atas:.3f}%")
print(f"Jumlah outlier: {mask_outlier.sum()} dari {len(r)} hari ({mask_outlier.sum() / len(r) * 100:.1f}%)")
print("\n10 return paling ekstrem:")
display(usd.loc[r.abs().sort_values(ascending=False).head(10).index, ["Date", "Price", "Return_%"]])

# Cek konsistensi harga (High harus >= Low, Price harus berada di antara Low dan High)
tidak_valid = ((usd["High"] < usd["Low"]) | (usd["Price"] > usd["High"]) | (usd["Price"] < usd["Low"])).sum()
print("\nBaris dengan harga tidak konsisten:", tidak_valid)

**Keputusan:** outlier return **dipertahankan** (tidak dihapus dan tidak di-clip).
**Alasan:** lonjakan besar pada USD/IDR adalah peristiwa pasar nyata (misalnya guncangan global), bukan salah input. Menghapusnya membuat model tidak pernah melihat kondisi stres, padahal hari seperti itulah yang paling sulit diprediksi. Untuk memastikan itu bukan salah input, harga diperiksa konsistensinya (`High >= Low`, `Price` di antara `Low` dan `High`). Dampak outlier pada KNN dan Logistic Regression dikurangi lewat **scaling** di dalam pipeline, sedangkan model berbasis pohon tidak sensitif terhadap outlier.

### 3.5 Encoding kolom kategorik

In [ ]:
print("Kolom bertipe object/kategori yang tersisa di USD/IDR:", usd.select_dtypes(include=["object", "category"]).columns.tolist())
print("Kolom tanggal:", usd.select_dtypes(include=["datetime64[ns]"]).columns.tolist())

**Keputusan:** setelah konversi tipe, tidak ada lagi kolom teks. Satu-satunya informasi kategorik adalah **hari dalam seminggu** (diturunkan dari `Date`), yang di-encode dengan **one-hot encoding** (`drop_first=True`) pada tahap feature engineering. `Date` sendiri tidak dipakai sebagai fitur.
**Alasan:** hari adalah kategori tanpa urutan makna (Senin bukan "lebih kecil" dari Jumat), sehingga label angka 0-6 akan menyesatkan model berbasis jarak seperti KNN. One-hot menghindari masalah itu, dan `drop_first=True` mencegah kolom yang redundan. `Date` tidak dipakai langsung karena nilai kalender yang terus bertambah tidak bermakna bagi data baru di masa depan.

## 4. EDA Lanjutan: Target dan Visualisasi

### 4.1 Membuat target

Target = 1 jika `Price` besok > `Price` hari ini, selain itu 0 (turun atau tetap). Baris terakhir dibuang karena belum punya harga hari berikutnya.

In [ ]:
usd["Target"] = (usd["Price"].shift(-1) > usd["Price"]).astype(int)
usd = usd.iloc[:-1].copy()

jumlah_tetap = (usd["Price"].shift(-1) == usd["Price"]).sum()
print(f"Hari dengan harga besok persis sama (masuk kelas 0): {jumlah_tetap}")

dist = usd["Target"].value_counts().sort_index()
display(pd.DataFrame({"Jumlah": dist, "Persen": (dist / dist.sum() * 100).round(2)}).rename_axis("Target"))

### 4.2 Visualisasi 1: Pergerakan harga USD/IDR

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(usd["Date"], usd["Price"], linewidth=1.2)
plt.title("Pergerakan Harian USD/IDR")
plt.xlabel("Tanggal")
plt.ylabel("USD/IDR")
plt.tight_layout()
plt.show()

print(f"Harga terendah {usd['Price'].min():,.0f} dan tertinggi {usd['Price'].max():,.0f}")

**Temuan 1:** kurs bergerak naik dalam jangka panjang (rupiah melemah) dari kisaran 13 ribu ke atas 17 ribu, dengan beberapa lonjakan tajam. Karena levelnya terus bergeser, **harga mentah tidak boleh dipakai sebagai fitur**. Level harga hanya akan menjadi "penanda waktu" dan tidak mengandung pola arah. Fitur yang dipakai adalah return, rasio, dan volatilitas yang stabil dari waktu ke waktu.

### 4.3 Visualisasi 2: Distribusi return harian dan outlier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(usd["Return_%"].dropna(), bins=60, kde=True, ax=axes[0])
axes[0].set_title("Distribusi Return Harian USD/IDR")
axes[0].set_xlabel("Return (%)")
sns.boxplot(x=usd["Return_%"].dropna(), ax=axes[1])
axes[1].set_title("Boxplot Return Harian")
axes[1].set_xlabel("Return (%)")
plt.tight_layout()
plt.show()

display(usd["Return_%"].describe().to_frame("Return_%").T)

**Temuan 2:** return harian terpusat di sekitar 0% dengan simpangan baku sekitar 0,37%, artinya pergerakan harian biasanya kecil. Ekor distribusi panjang: ada hari ekstrem sekitar -2,7% dan +4,6%. Ini mendukung keputusan mempertahankan outlier sebagai kejadian pasar nyata, dan menunjukkan bahwa arah harian sangat berisik (noise tinggi).

### 4.4 Visualisasi 3: Distribusi target

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=usd, x="Target")
ax.set_xticklabels(["0 = Turun/Tetap", "1 = Naik"])
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.title("Distribusi Target Arah USD/IDR")
plt.xlabel("")
plt.ylabel("Jumlah hari")
plt.tight_layout()
plt.show()

**Temuan 3:** kelas cukup seimbang, dengan kelas "Turun/Tetap" sedikit lebih banyak (sekitar 53% dibanding 47%). Konsekuensinya: model yang selalu menebak "Turun/Tetap" sudah mendapat accuracy sekitar 53%, jadi **53% adalah batas bawah yang harus dilampaui** agar sebuah model dianggap punya kemampuan prediksi. Karena ketidakseimbangannya ringan, resampling tidak diperlukan; cukup `stratify=y` saat split agar proporsi kelas terjaga.

## 5. Feature Engineering

Semua fitur untuk baris hari *t* hanya memakai informasi sampai hari *t*, sedangkan target adalah arah hari *t+1*, sehingga tidak ada kebocoran dari masa depan.

- **Return multi-periode** (1, 3, 5, 10 hari), **posisi harga terhadap moving average** (5, 10, 20 hari), dan **volatilitas** rolling (5, 10, 20 hari).
- **Bentuk candle hari ini:** rentang harian, selisih harga tutup-buka, dan gap pembukaan terhadap penutupan kemarin.
- **Variabel eksternal:** return JCI dan DXY (1 dan 5 hari). DXY ikut dipakai karena datanya lengkap (sekitar 2.800 baris, hampir sama dengan USD/IDR).
- **Hari dalam seminggu** (one-hot, lihat 3.5).

In [ ]:
data = usd[["Date", "Price", "Open", "High", "Low", "Target"]].copy()

# Gabungkan data eksternal berdasarkan tanggal, lalu forward fill (keputusan 3 di bagian 3.2)
for nama, df in [("JCI", jci), ("DXY", dxy)]:
    data = data.merge(df[["Date", "Price"]].rename(columns={"Price": f"{nama}_Price"}), on="Date", how="left")
print("Missing sebelum ffill:", data[["JCI_Price", "DXY_Price"]].isna().sum().to_dict())
data[["JCI_Price", "DXY_Price"]] = data[["JCI_Price", "DXY_Price"]].ffill()
print("Missing setelah ffill:", data[["JCI_Price", "DXY_Price"]].isna().sum().to_dict())

ret = data["Price"].pct_change()
for k in [1, 3, 5, 10]:
    data[f"Return_{k}"] = data["Price"].pct_change(k)
for w in [5, 10, 20]:
    data[f"Price_vs_MA{w}"] = data["Price"] / data["Price"].rolling(w).mean() - 1
    data[f"Volatility_{w}"] = ret.rolling(w).std()

data["Intraday_Range"] = (data["High"] - data["Low"]) / data["Price"]
data["Close_vs_Open"] = data["Price"] / data["Open"] - 1
data["Gap_Open"] = data["Open"] / data["Price"].shift(1) - 1

for nama in ["JCI", "DXY"]:
    data[f"{nama}_Return_1"] = data[f"{nama}_Price"].pct_change(1)
    data[f"{nama}_Return_5"] = data[f"{nama}_Price"].pct_change(5)

# One-hot encoding hari dalam seminggu
hari = pd.get_dummies(data["Date"].dt.day_name(), prefix="Hari", drop_first=True, dtype=int)
data = pd.concat([data, hari], axis=1)

fitur_numerik = [
    "Return_1", "Return_3", "Return_5", "Return_10",
    "Price_vs_MA5", "Price_vs_MA10", "Price_vs_MA20",
    "Volatility_5", "Volatility_10", "Volatility_20",
    "Intraday_Range", "Close_vs_Open", "Gap_Open",
    "JCI_Return_1", "JCI_Return_5", "DXY_Return_1", "DXY_Return_5",
]
fitur_hari = hari.columns.tolist()
features = fitur_numerik + fitur_hari

sebelum = len(data)
model_data = data[["Date", "Target"] + features].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
print(f"Baris dibuang karena NaN akibat rolling/shift di awal data: {sebelum - len(model_data)}")
print("Jumlah fitur:", len(features))
display(model_data.head())

**Catatan keputusan:** baris awal yang NaN akibat jendela rolling dibuang (bukan diisi) karena jumlahnya kecil (sekitar 20 baris dari lebih dari 2.800) dan nilai pengisinya tidak punya dasar. Fitur level harga mentah (`Open`, `High`, `Low`, `Price`, MA) **tidak dipakai** sesuai Temuan 1.

### 5.1 Visualisasi 4: Hubungan fitur dengan target

In [ ]:
korelasi = model_data[features + ["Target"]].corr()["Target"].drop("Target").sort_values()

plt.figure(figsize=(8, 7))
korelasi.plot(kind="barh")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Korelasi Fitur dengan Target (arah besok)")
plt.xlabel("Korelasi Pearson")
plt.tight_layout()
plt.show()

print(f"Korelasi absolut terbesar: {korelasi.abs().max():.3f} ({korelasi.abs().idxmax()})")

# Pasangan fitur yang sangat berkorelasi satu sama lain
kor_fitur = model_data[fitur_numerik].corr().abs()
pasangan = (kor_fitur.where(np.triu(np.ones(kor_fitur.shape, dtype=bool), k=1)).stack().sort_values(ascending=False))
print("\nPasangan fitur dengan korelasi > 0.8:")
display(pasangan[pasangan > 0.8].to_frame("Korelasi absolut"))

**Temuan 4:** sebagian besar fitur berkorelasi lemah dengan target (|r| di bawah sekitar 0,1). Pengecualiannya adalah **`DXY_Return_1` (r = 0,259)**, disusul `DXY_Return_5` (sekitar 0,11) dan `Close_vs_Open` (sekitar 0,07). Perubahan indeks dolar AS hari ini jadi petunjuk paling informatif untuk arah USD/IDR besok. Dugaan penyebabnya: pasar AS tutup setelah pasar Asia, sehingga pergerakan DXY di sesi AS belum tercermin pada kurs USD/IDR hari yang sama. Ini dugaan, bukan hasil yang diuji. Korelasi 0,26 berarti sekitar 7% variasi target (r kuadrat), jadi sinyalnya nyata tetapi sedang, bukan kuat. Hubungan linear juga belum tentu berarti model mampu memanfaatkannya, sehingga tetap dibandingkan dengan Dummy. Beberapa fitur saling berkorelasi sangat tinggi (misalnya `Return_3` dan `Price_vs_MA5`, 0,96). Hal ini tidak masalah bagi model pohon, tetapi perlu diingat saat menafsirkan koefisien Logistic Regression dan pada KNN yang berbasis jarak.

## 6. Split Data

Sesuai ketentuan: `train_test_split` dengan `test_size=0.2`, `random_state=42`, dan `stratify=y`.
Data uji **disimpan dan tidak disentuh** sampai tahap evaluasi akhir.

In [ ]:
X = model_data[features]
y = model_data["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Proporsi kelas 1 pada data total :", round(y.mean(), 4))
print("Proporsi kelas 1 pada data latih :", round(y_train.mean(), 4))
print("Proporsi kelas 1 pada data uji   :", round(y_test.mean(), 4))

> **Catatan metodologis:** ketentuan mewajibkan split acak. Pada data deret waktu, split acak membuat baris latih dan uji saling berselang-seling dan jendela rolling-nya saling tumpang tindih, sehingga skor uji bisa sedikit lebih optimistis daripada kondisi nyata (memprediksi masa depan murni). Hal ini dibahas di bagian Keterbatasan.

## 7. Model Standar: KNeighborsClassifier dengan Scaling

KNN berbasis jarak, sehingga fitur dengan skala besar mendominasi bila tidak diseragamkan. Karena itu `StandardScaler` dipasang **di dalam `Pipeline`**: pada setiap lipatan cross-validation, scaler hanya belajar dari data latih lipatan itu (tidak bocor ke data validasi).

Model ini menjadi **pembanding** bagi model lain. Selain itu ditambahkan `DummyClassifier` (selalu menebak kelas terbanyak) sebagai batas bawah.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

knn_baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])

## 8. Model Lain (Opsional)

Logistic Regression, Random Forest, XGBoost, dan LightGBM. Semuanya memakai **data latih, fitur, dan skema cross-validation yang sama** dengan model standar. Logistic Regression diberi scaling karena sensitif terhadap skala fitur. Model berbasis pohon tidak memerlukannya.

In [ ]:
models_awal = {
    "Dummy (kelas terbanyak)": DummyClassifier(strategy="most_frequent"),
    "KNN (standar)": knn_baseline,
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, random_state=RANDOM_STATE, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE, verbosity=-1),
}

## 9. Pemilihan Model: Cross-Validation 5 Lipatan dan Tuning

Semua perbandingan dan tuning memakai **hanya data latih** dengan `StratifiedKFold(n_splits=5)`. Data uji belum dipakai.

### 9.1 Perbandingan awal (hyperparameter default)

Tabel menampilkan skor **data latih** dan skor **validasi (rata-rata 5 lipatan)** berdampingan. Selisih yang besar menandakan overfitting.

In [ ]:
scoring = ["accuracy", "precision", "recall"]

def ringkas_cv(nama, model):
    hasil = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring,
                           return_train_score=True, n_jobs=-1)
    baris = {"Model": nama}
    for m in scoring:
        baris[f"{m}_train"] = hasil[f"train_{m}"].mean()
        baris[f"{m}_cv"] = hasil[f"test_{m}"].mean()
    baris["accuracy_cv_std"] = hasil["test_accuracy"].std()
    return baris

cv_awal = pd.DataFrame([ringkas_cv(n, m) for n, m in models_awal.items()]).set_index("Model")
display(cv_awal.round(4))

Perhatikan dua hal pada tabel: (1) apakah accuracy CV melampaui baris *Dummy*, dan (2) selisih skor latih dan CV (model pohon biasanya sangat tinggi di data latih tetapi turun di validasi).

### 9.2 Tuning hyperparameter dengan GridSearchCV (5 lipatan)

Pencarian parameter memakai data latih saja dan `scoring="accuracy"`.

In [ ]:
param_grids = {
    "KNN (standar)": (models_awal["KNN (standar)"], {
        "knn__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
        "knn__weights": ["uniform", "distance"],
        "knn__p": [1, 2],
    }),
    "Logistic Regression": (models_awal["Logistic Regression"], {
        "lr__C": [0.001, 0.01, 0.1, 1, 10],
    }),
    "Random Forest": (models_awal["Random Forest"], {
        "max_depth": [3, 5, 8],
        "min_samples_leaf": [5, 10, 20],
    }),
    "XGBoost": (models_awal["XGBoost"], {
        "n_estimators": [100, 300],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.03, 0.1],
    }),
    "LightGBM": (models_awal["LightGBM"], {
        "n_estimators": [100, 300],
        "num_leaves": [7, 15],
        "learning_rate": [0.03, 0.1],
    }),
}

hasil_tuning = {}
for nama, (model, grid) in param_grids.items():
    gs = GridSearchCV(model, grid, cv=cv, scoring="accuracy", n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    hasil_tuning[nama] = gs
    print(f"{nama:22s} | CV accuracy terbaik = {gs.best_score_:.4f} | {gs.best_params_}")

### 9.3 Perbandingan model setelah tuning

In [ ]:
baris = [ringkas_cv("Dummy (kelas terbanyak)", models_awal["Dummy (kelas terbanyak)"])]
for nama, gs in hasil_tuning.items():
    r_ = ringkas_cv(nama + " (tuned)", gs.best_estimator_)
    baris.append(r_)

cv_tuned = pd.DataFrame(baris).set_index("Model")
cv_tuned["selisih_acc_train_cv"] = cv_tuned["accuracy_train"] - cv_tuned["accuracy_cv"]
display(cv_tuned.round(4))

ax = cv_tuned[["accuracy_train", "accuracy_cv"]].plot(kind="bar", figsize=(11, 5))
ax.axhline(cv_tuned.loc["Dummy (kelas terbanyak)", "accuracy_cv"], color="red", linestyle="--", label="Batas bawah (Dummy)")
ax.set_ylim(0.4, 1.0)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy Latih vs Cross-Validation (setelah tuning)")
plt.xticks(rotation=25, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

### 9.4 Memilih model

**Aturan pemilihan (ditetapkan sebelum melihat data uji):** model dengan **accuracy cross-validation tertinggi** dari hasil tuning (tidak termasuk Dummy). Precision dan recall CV ditampilkan untuk memastikan model tidak sekadar menebak satu kelas.

In [ ]:
kandidat = cv_tuned.drop(index="Dummy (kelas terbanyak)")
nama_terpilih_tuned = kandidat["accuracy_cv"].idxmax()
nama_terpilih = nama_terpilih_tuned.replace(" (tuned)", "")
model_terpilih = hasil_tuning[nama_terpilih].best_estimator_

print("Model terpilih :", nama_terpilih)
print("Parameter      :", hasil_tuning[nama_terpilih].best_params_)
print(f"Accuracy CV    : {kandidat.loc[nama_terpilih_tuned, 'accuracy_cv']:.4f}")
print(f"Accuracy CV KNN (standar, tuned): {kandidat.loc['KNN (standar) (tuned)', 'accuracy_cv']:.4f}")

## 10. Evaluasi Akhir pada Data Uji

Data uji dipakai **satu kali**, hanya untuk model terpilih (yang sudah di-fit ulang pada seluruh data latih oleh `GridSearchCV`). Model lain dan KNN pembanding dibandingkan lewat cross-validation di atas, bukan lewat data uji, agar data uji tetap bersih dari proses memilih model.

In [ ]:
def metrik(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
    }

pred_train = model_terpilih.predict(X_train)
pred_test = model_terpilih.predict(X_test)   # satu-satunya penggunaan data uji

skor_akhir = pd.DataFrame({
    "Data Latih": metrik(y_train, pred_train),
    "Data Uji": metrik(y_test, pred_test),
}).T

print(f"Model: {nama_terpilih}")
display(skor_akhir.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (judul, yt, yp) in zip(axes, [("Data Latih", y_train, pred_train), ("Data Uji", y_test, pred_test)]):
    sns.heatmap(confusion_matrix(yt, yp), annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Turun/Tetap", "Naik"], yticklabels=["Turun/Tetap", "Naik"])
    ax.set_title(f"Confusion Matrix - {judul}")
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")
plt.tight_layout()
plt.show()

## 11. Interpretasi

In [ ]:
acc_tr, acc_te = skor_akhir.loc["Data Latih", "Accuracy"], skor_akhir.loc["Data Uji", "Accuracy"]
pre_te, rec_te = skor_akhir.loc["Data Uji", "Precision"], skor_akhir.loc["Data Uji", "Recall"]
acc_dummy = max(y_train.mean(), 1 - y_train.mean())
tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()

print(f"MODEL TERPILIH: {nama_terpilih}\n")
print("ARTI METRIK (data uji, bahasa awam)")
print(f"- Accuracy {acc_te:.1%}: dari 100 hari, tebakan arah benar pada sekitar {acc_te*100:.0f} hari.")
print(f"  Sebagai pembanding, menebak 'Turun/Tetap' terus-menerus sudah benar pada sekitar {acc_dummy*100:.0f} hari.")
print(f"- Precision {pre_te:.1%}: setiap model bilang 'besok naik', hanya sekitar {pre_te*100:.0f} dari 100 peringatan itu yang benar.")
print(f"- Recall {rec_te:.1%}: dari semua hari yang benar-benar naik, model berhasil menangkap sekitar {rec_te*100:.0f} dari 100.")
print(f"- Confusion matrix: {tp} hari naik tertebak benar, {fn} hari naik terlewat, {fp} alarm naik palsu, {tn} hari turun/tetap tertebak benar.")

print("\nKESIMPULAN KEMAMPUAN MODEL")
if acc_te - acc_dummy < 0.02:
    print(f"- Accuracy uji hanya selisih {100*(acc_te-acc_dummy):+.1f} poin dari tebakan kelas terbanyak, jadi model BELUM terbukti punya kemampuan prediksi arah yang berarti.")
else:
    print(f"- Accuracy uji unggul {100*(acc_te-acc_dummy):+.1f} poin dari tebakan kelas terbanyak, tetapi selisih sekecil ini perlu dikonfirmasi pada data waktu yang benar-benar baru.")
gap = acc_tr - acc_te
if gap > 0.10:
    print(f"- Accuracy latih ({acc_tr:.1%}) jauh di atas uji ({acc_te:.1%}): tanda overfitting, model menghafal data latih.")
else:
    print(f"- Accuracy latih ({acc_tr:.1%}) dan uji ({acc_te:.1%}) berdekatan: tidak ada tanda overfitting yang kuat.")

### Alasan memilih model

Model dipilih berdasarkan **accuracy cross-validation 5 lipatan tertinggi pada data latih** setelah tuning, dengan aturan yang ditetapkan sebelum data uji dibuka. Semua model memakai fitur, data latih, dan skema CV yang sama, sehingga perbandingannya adil. KNN dengan scaling tetap menjadi pembanding standar: model lain hanya layak dipilih jika accuracy CV-nya lebih baik, dan tambahan kompleksitasnya (pohon boosting, banyak parameter) terbayar dengan peningkatan nyata. Bila selisihnya kecil, model yang lebih sederhana lebih mudah dipertanggungjawabkan.

### Keterbatasan hasil

1. **Sinyal prediksi lemah sampai sedang.** Arah harian kurs dipengaruhi berita dan kejadian yang tidak ada di dataset. Hanya DXY yang berkorelasi cukup jelas dengan target (Temuan 4, r = 0,26), sehingga wajar bila accuracy hanya sedikit di atas batas bawah. Skor yang sedang dengan proses yang benar lebih jujur daripada skor tinggi yang berasal dari kebocoran data.
2. **Split acak pada data deret waktu.** Ketentuan mewajibkan split acak dengan `stratify`. Akibatnya hari-hari yang bertetangga bisa terbagi ke data latih dan data uji, dan jendela rolling-nya tumpang tindih, sehingga skor uji berpotensi **lebih optimistis** daripada prediksi masa depan sesungguhnya. Evaluasi yang lebih ketat memakai split berdasarkan waktu atau `TimeSeriesSplit`.
3. **Data uji kecil.** Sekitar 560 hari saja, sehingga selisih beberapa poin accuracy bisa hanya kebetulan statistik.
4. **Fitur terbatas.** Hanya harga USD/IDR, JCI, dan DXY. Suku bunga, data inflasi, dan berita tidak dimasukkan.
5. **Bukan alat trading.** Accuracy tidak memperhitungkan besar pergerakan, biaya transaksi, atau selisih jual-beli. Model ini tidak boleh dijadikan dasar keputusan keuangan.